In [1]:
import myflopy as mf
import figs as f
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
mf.TriangleGrid

myflopy.modflow.mf6.grid.triangle.TriangleGrid

In [5]:
mf.Project('test_project').save()


PosixPath('test_project/specs/project_spec.json')

In [2]:
project = mf.ProjectSpec('test_project')
project.save()

AttributeError: module 'myflopy' has no attribute 'ProjectSpec'

In [ ]:
mf.ghb()

grid = mf.GridSpec.voronoi(
    name="voronoi",
    boundary=mf.GeoPackageSourceSpec(
        "inputs/test_data.gpkg",
        layer="domain",
    ),
    refinement=mf.GeoPackageSourceSpec(
        "inputs/grid_inputs.gpkg",
        layer="refinement",
        fields={
            "area": "max_area",
            "label": "name",
            "priority": "priority",
        },
    ),
    breaklines=[
        mf.GeoPackageSourceSpec(
            "inputs/grid_inputs.gpkg",
            layer="streams",
            fields={
                "area": "max_area",
                "label": "name",
            },
        )
    ],
    points=[
        mf.GeoPackageSourceSpec(
            "inputs/grid_inputs.gpkg",
            layer="wells",
        )
    ],
    crs="EPSG:2927",
    boundary_max_area=20_000,
    #breakline_buffer=50,
    #profile="clean",
)

gwf = (
    mf.gwf("gwf")
    .with_grid(grid)
    .with_package(mf.ic(strt=100.0))
    .with_package(mf.npf(k=25.0))
)

baseline = mf.SimulationSpec(
    "baseline",
    models=(gwf,),
    packages=(
        mf.tdis(nper=1, perioddata=[(1.0, 1, 1.0)]),
    ),
)

project = project.with_simulation(baseline)

project.save()

run = project.build("baseline")

model = run.flopy_model("gwf")
resolved_grid = model.myflopy_context.grid

print(run.workspace)
print(resolved_grid.ncpl)
print(resolved_grid.gdf_vorPolys.head())